# 05 — The powered ladder: multi-seed × multi-algorithm sweep

Multi-seed × multi-algorithm rerun of notebook 04's ladder, built to run long on the A100.
One seed on the laptop was enough to *build* the protocol; this notebook is what makes the
verdict *defensible*: mean ± std over seeds, plus the non-IID-aware aggregators.

**Pre-registered decisions (fixed BEFORE the big run — do not tune after):**
1. **Network frozen** at the config's 35 trusts, even on the 8-year dataset — extended history
   deepens each client's series; it does not re-select clients.
2. **Split boundaries are dates, not counts** — train ≤ 2024-09, val ≤ 2025-03, test ≤ 2026-03.
   With 96 months (2018-04→2026-03) the train window grows 30 → 66 target months; val/test are
   unchanged, so results stay comparable across dataset sizes.
3. **Seeds 0–4**, all algorithms, identical protocol per rung (val-selected where applicable).
4. Algorithms: local-only · centralized · FedAvg · **FedProx** (μ=0.1) · **FedAdam**
   (server lr 0.1) · each fed variant + val-selected fine-tune.
5. Metric: MAE in referral units, mean per-trust and pooled; selection on val only.

**To run on the A100:** run notebooks 00-04 first (04 builds `features.csv`), set `SMOKE = False` below, then
`jupyter nbconvert --to notebook --execute --inplace 06_a100_sweep.ipynb` inside tmux.
Results append to `sweep_results.jsonl` after every run — killing and rerunning resumes
where it stopped.

In [ ]:
SMOKE = False  # FULL sweep (5 seeds x 8 algorithms). Set True only to smoke-test the plumbing.

SEEDS = [0] if SMOKE else [0, 1, 2, 3, 4]
ROUNDS = 15 if SMOKE else 100          # federated rounds
EPOCHS = 10 if SMOKE else 40           # local-only / centralized / fine-tune epoch budget
PROX_MU = 0.1                          # FedProx proximal strength (pre-registered)
SERVER_LR = 0.1                        # FedAdam server step size (pre-registered)
OUT = "../data/processed/sweep_results.jsonl"
print(f"SMOKE={SMOKE} | seeds={SEEDS} | rounds={ROUNDS} | epochs={EPOCHS}")

In [ ]:
# data + model + helpers (self-contained; mirrors notebook 04)
import json
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

f = pd.read_csv("../data/processed/features.csv")
FEATS = ["lag1", "lag2", "lag3", "lag12", "backlog", "throughput", "month_sin", "month_cos"]
specs = sorted(f["specialty_code"].unique())
f["spec"] = f["specialty_code"].map({s: i for i, s in enumerate(specs)})
trusts = sorted(f["provider_code"].unique())
splits = {s: f[f["split"] == s].reset_index(drop=True) for s in ["train", "val", "test"]}
device = "cuda" if torch.cuda.is_available() else "cpu"
te = splits["test"]
client_train = {t: splits["train"][splits["train"]["provider_code"] == t] for t in trusts}
client_val = {t: splits["val"][splits["val"]["provider_code"] == t] for t in trusts}

def tensors(rows):
    return (torch.tensor(rows[FEATS].values, dtype=torch.float32).to(device),
            torch.tensor(rows["spec"].values).to(device),
            torch.tensor(rows["y"].values, dtype=torch.float32).to(device))

class Net(nn.Module):
    def __init__(self, emb=4, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(len(specs), emb)
        self.mlp = nn.Sequential(nn.Linear(len(FEATS) + emb, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x, s):  # residual on lag1 -> starts at the naive baseline
        return x[:, 0] + self.mlp(torch.cat([x, self.emb(s)], dim=1)).squeeze(-1)

def train_epochs(model, rows, epochs, lr=1e-3, batch=256, prox_ref=None, mu=0.0):
    x, s, y = tensors(rows)
    dl = DataLoader(TensorDataset(x, s, y), batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for xb, sb, yb in dl:
            opt.zero_grad()
            loss = nn.functional.l1_loss(model(xb, sb), yb)
            if prox_ref is not None:  # FedProx: keep local weights near the global reference
                loss = loss + (mu / 2) * sum(((p - g) ** 2).sum()
                                             for p, g in zip(model.parameters(), prox_ref))
            loss.backward()
            opt.step()
    return model

@torch.no_grad()
def predict_demand(model, rows):
    x, s, _ = tensors(rows)
    model.eval()
    ylog = model(x, s).cpu().numpy() * rows["sd"].values + rows["mu"].values
    return np.expm1(ylog)

def mae(rows, pred):
    err = np.abs(pred - rows["demand"].values)
    return pd.Series(err).groupby(rows["provider_code"].values).mean().mean(), err.mean()

def fit_with_val(rows_tr, rows_va, epochs, lr=1e-3, seed=0, init=None):
    torch.manual_seed(seed)
    model = Net().to(device)
    if init is not None:
        model.load_state_dict(init)
    best, best_m = None, np.inf
    for _ in range(epochs):
        train_epochs(model, rows_tr, 1, lr)
        m, _ = mae(rows_va, predict_demand(model, rows_va))
        if m < best_m:
            best_m, best = m, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best)
    return model

print(f"device: {device} | rows: {len(f)} | trusts: {len(trusts)} "
      f"| train months: {splits['train']['month'].nunique()}")

In [ ]:
# one federated trainer covering FedAvg / FedProx / FedAdam
def fed_train(agg="avg", mu=0.0, rounds=ROUNDS, local_epochs=2, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    global_model = Net().to(device)
    weights = {t: len(client_train[t]) / len(splits["train"]) for t in trusts}
    m_state = v_state = None  # FedAdam server moments
    best, best_m = None, np.inf
    for r in range(rounds):
        gsd = global_model.state_dict()
        prox_ref = [p.detach().clone() for p in global_model.parameters()] if mu > 0 else None
        avg = None
        for t in trusts:  # each client trains from global weights on ITS OWN rows only
            local = Net().to(device)
            local.load_state_dict(gsd)
            train_epochs(local, client_train[t], local_epochs, lr, prox_ref=prox_ref, mu=mu)
            sd = local.state_dict()
            avg = {k: weights[t] * v for k, v in sd.items()} if avg is None \
                  else {k: avg[k] + weights[t] * sd[k] for k in avg}
        if agg == "adam":  # server-side Adam on the pseudo-gradient (avg - global)
            delta = {k: avg[k] - gsd[k] for k in avg}
            m_state = delta if m_state is None else {k: 0.9 * m_state[k] + 0.1 * delta[k] for k in delta}
            v_state = {k: delta[k] ** 2 for k in delta} if v_state is None \
                      else {k: 0.99 * v_state[k] + 0.01 * delta[k] ** 2 for k in delta}
            new = {k: gsd[k] + SERVER_LR * m_state[k] / (v_state[k].sqrt() + 1e-3) for k in delta}
        else:              # plain weighted average
            new = avg
        global_model.load_state_dict(new)
        m, _ = mae(splits["val"], predict_demand(global_model, splits["val"]))
        if m < best_m:
            best_m, best = m, {k: v.clone() for k, v in new.items()}
    global_model.load_state_dict(best)
    return global_model

def finetune_eval(fed_model, seed):  # identical protocol to local-only; init = fed weights
    pred = np.zeros(len(te))
    for t in trusts:
        m = fit_with_val(client_train[t], client_val[t], EPOCHS, seed=seed, init=fed_model.state_dict())
        mask = (te["provider_code"] == t).values
        pred[mask] = predict_demand(m, te[mask])
    return mae(te, pred)

In [ ]:
# the sweep: every (seed, algorithm); each result appends to jsonl immediately -> resumable
done = set()
if os.path.exists(OUT):
    with open(OUT) as fh:
        done = {(json.loads(l)["seed"], json.loads(l)["algo"]) for l in fh if l.strip()}
print(f"already done: {len(done)} runs")

def record(seed, algo, res):
    with open(OUT, "a") as fh:
        fh.write(json.dumps({"seed": seed, "algo": algo, "smoke": SMOKE,
                             "per_trust_mae": round(res[0], 2), "pooled_mae": round(res[1], 2)}) + "\n")
    print(f"seed {seed} | {algo:<12} per-trust MAE {res[0]:.1f}")

for seed in SEEDS:
    if (seed, "local") not in done:
        pred = np.zeros(len(te))
        for t in trusts:
            m = fit_with_val(client_train[t], client_val[t], EPOCHS, seed=seed)
            mask = (te["provider_code"] == t).values
            pred[mask] = predict_demand(m, te[mask])
        record(seed, "local", mae(te, pred))
    if (seed, "central") not in done:
        m = fit_with_val(splits["train"], splits["val"], EPOCHS, seed=seed)
        record(seed, "central", mae(te, predict_demand(m, te)))
    for algo, kw in [("fedavg", {}), ("fedprox", {"mu": PROX_MU}), ("fedadam", {"agg": "adam"})]:
        if (seed, algo) in done and (seed, algo + "_ft") in done:
            continue
        fed = fed_train(seed=seed, **kw)
        if (seed, algo) not in done:
            record(seed, algo, mae(te, predict_demand(fed, te)))
        if (seed, algo + "_ft") not in done:
            record(seed, algo + "_ft", finetune_eval(fed, seed))
print("sweep complete")

In [ ]:
# summary: mean +- std over seeds (only meaningful with SMOKE=False and all 5 seeds)
import matplotlib.pyplot as plt
res = pd.DataFrame([json.loads(l) for l in open(OUT) if l.strip()])
res = res[res["smoke"] == SMOKE]
summary = res.groupby("algo")["per_trust_mae"].agg(["mean", "std", "count"]).round(2) \
             .sort_values("mean")
print(summary.to_string())

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.labelcolor": "#52514e", "xtick.color": "#898781", "ytick.color": "#898781",
})
fig, ax = plt.subplots(figsize=(7, 3.2))
s = summary.iloc[::-1]
ax.barh(s.index, s["mean"], xerr=s["std"].fillna(0), color="#2a78d6", height=0.6,
        error_kw={"ecolor": "#52514e", "capsize": 3})
for i, v in enumerate(s["mean"].values):
    ax.text(v + s["mean"].max() * 0.012, i, f"{v:.1f}", va="center", fontsize=8, color="#0b0b0b")
ax.set_title(f"Sweep: test MAE, mean ± std over {res['seed'].nunique()} seed(s)")
ax.set_xlabel("per-trust MAE (referrals)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()